# Código do Luís (Ollama) a correr no Colab

Corre o `run_inference_LangChain_HF.py` **sem alterações**: instala o Ollama dentro da máquina do Colab, descarrega o
mesmo modelo e usa o script como está. Antes: **Runtime → Change runtime type → GPU (T4)**, senão o Ollama usa o
processador e fica lento.

In [1]:
!nvidia-smi

Mon Nov 16 23:58:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Branch publicada no GitHub (git push). O código do Luís tem de estar na branch.
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"
MODEL = "qwen3.5:4b"   # o mesmo modelo do README

import os
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
os.chdir("/content/kl")
!git log --oneline -1

Cloning into 'kl'...
remote: Enumerating objects: 745, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 745 (delta 220), reused 245 (delta 160), pack-reused 408 (from 1)
Receiving objects: 100% (745/745), 40.60 MiB | 15.53 MiB/s, done.
Resolving deltas: 100% (532/532), done.
Updating files: 100% (454/454), done.
dc3d4f1 (HEAD -> rui/colab, origin/rui/colab) Inferência local com transformers para correr no Colab


In [3]:
# Instalar o Ollama na máquina do Colab, arrancar o servidor e descarregar o modelo
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("O servidor do Ollama não arrancou. Vê /content/ollama.log")
!ollama pull {MODEL}

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,120 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Addi

In [4]:
# Dependências do script (sem o requirements.txt do repositório)
!pip install -q langchain-ollama tqdm retry

## 1. Teste com 3 prompts
O script percorre todos os ficheiros da pasta de entrada, por isso criamos pastas só com o ficheiro que queremos.

In [7]:
import json
SRC = "1_data_module/2_creating_prompts/data_with_prompts/three_shot_RAG_prompts_test.json"
os.makedirs("/content/prompts_teste", exist_ok=True)
json.dump(json.load(open(SRC))[:3], open("/content/prompts_teste/three_shot_RAG_prompts_test.json", "w"))
!python 2_inference_module/run_inference_LangChain_HF.py --input_folder /content/prompts_teste --model_name {MODEL} --out_dir /content/out_teste

Traceback (most recent call last):
  File "/content/kl/2_inference_module/run_inference_LangChain_HF.py", line 1, in <module>
    from langchain_huggingface import HuggingFaceEndpoint
ModuleNotFoundError: No module named 'langchain_huggingface'


In [6]:
for x in json.load(open("/content/out_teste/three_shot_RAG_prompts_test.json")):
    print(repr(x["generated_text"][:300]))

FileNotFoundError: [Errno 2] No such file or directory: '/content/out_teste/three_shot_RAG_prompts_test.json'

## 2. Corrida completa (446 prompts, só o three_shot_RAG)
O script só grava no fim de cada ficheiro. Se o Colab desligar a meio, perde-se esta corrida. Guardamos o resultado no Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kl_out_ollama"

os.makedirs("/content/prompts_rag", exist_ok=True)
!cp {SRC} /content/prompts_rag/
!python 2_inference_module/run_inference_LangChain_HF.py --input_folder /content/prompts_rag --model_name {MODEL} --out_dir {OUT}

## 3. Post-processamento e avaliação

In [ ]:
!python 2_inference_module/post_process_generated_string.py --input_folder {OUT} --out_dir {OUT}_pp
!python 3_evaluation_module/evaluation_reference_to_annotations/evaluate.py --input_dir {OUT}_pp --gold_file 1_data_module/1_data_preprocessing/RED-fm/test.jsonl --out_dir {OUT}_eval